# Brain-expressed CNV overlap with SCHEMA and schizophrenia GWAS

Run top to bottom in a fresh project kernel. `PCNET2.2_brain` is the article
network; set `CNV_NETWORK=STRING_high_conf_brain` for the supporting interactome check.

```bash
.venv/bin/python scripts/run_network_notebooks.py --networks PCNET2.2_brain --notebooks cnv_brain_disorder_overlap
```

Results go to `results/<network>/`; each executed notebook and log has a unique
attempt directory under `results/executions/`. The minimum retained seed count
is set in the next cell. See the [notebook guide](README.md) and
[methods](../docs/methods.md).

Brain filtering uses >1 TPM in at least one of 13 HPA regions, then the largest
connected component. Inspect the printed seed QC.


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cnv_overlap.pipeline import AnalysisConfig, run
from cnv_overlap.visualization import pair_heatmap


ROOT = Path.cwd()
if ROOT.name == "notebooks":  # opened from notebooks/ in Jupyter
    ROOT = ROOT.parent
if not (ROOT / "configs").is_dir():
    raise RuntimeError("Start this notebook with the project root as working directory")
CONFIG_PATH = Path(os.environ.get("CNV_CONFIG", ROOT / "configs/cnv_disorder_overlap.json"))
import sys
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from network_inputs import load_network_inputs
from results_layout import results_dir
# Select PCNET2.2_brain or STRING_high_conf_brain.
NETWORK = os.environ.get("CNV_NETWORK", "PCNET2.2_brain")
if not NETWORK.endswith("_brain"):
    NETWORK += "_brain"
# Minimum retained seeds per gene set.
MIN_SEEDS = 3
spec, root, inputs, provenance = load_network_inputs(CONFIG_PATH, NETWORK, min_seeds=MIN_SEEDS)
RESULTS_ROOT = root / spec["results_root"]
print("Network:", NETWORK or "custom snapshot", "Results:", RESULTS_ROOT, "Cache:", root / spec["cache_dir"])
config = AnalysisConfig.from_dict(spec["analysis"])
if config.comparisons != "cnv_disorder":
    raise ValueError("This notebook requires analysis.comparisons=cnv_disorder")
A, nodes, symbols, entities, coordinates, regions = inputs
print(f"{len(nodes):,} nodes, {A.nnz // 2:,} edges; {len(entities)} gene sets")
print("Primary: overlap size; NetColoc normal-tail p-values; overlap policy:", config.overlap)
print("Gene-label permutations:", config.permutation_reps)

from notebook_analysis import SCZ_TARGETS, disorder_gene_partition


In [ ]:
# One run gives one BH family across every CNV–target pair for each statistic.
entities = {name: entity for name, entity in entities.items()
            if entity["kind"] == "cnv" or (entity["kind"] == "target" and name in SCZ_TARGETS)}
if not any(e["kind"] == "cnv" for e in entities.values()) or not any(e["kind"] == "target" for e in entities.values()):
    raise ValueError("Need CNVs and at least one target set")
HAS_SCZ = any(name in entities and entities[name]["kind"] == "target" for name in SCZ_TARGETS)
FAMILY_INPUTS = {"combined": (A, nodes, symbols, entities, coordinates, regions)}
RUNS = {"combined": run(*FAMILY_INPUTS["combined"], config=config,
    output_root=root/spec["output_root"], cache_dir=root/spec["cache_dir"],
    provenance={**provenance, "question_family": "combined",
                "included_targets": list(SCZ_TARGETS),
                "analysis_status": "brain-expressed SCHEMA/GWAS comparisons"})}
for family, run_dir in RUNS.items():
    assert json.loads((run_dir / "manifest.json").read_text())["status"] == "complete"
    print(family, run_dir)
    print((run_dir / "summary.md").read_text())
from cnv_overlap.results import notebook_execution_dir
FIGURES = notebook_execution_dir(RESULTS_ROOT, NETWORK) / "reports" / "cnv_brain_disorder_overlap"
FIGURES.mkdir(parents=True, exist_ok=True)
if not HAS_SCZ:
    print("No schema/gwaspri sets: these custom inputs have secondary comparisons only.")


## Input QC and tested hypotheses

Inspect seed losses, coordinate coverage and the hypothesis list. Coordinates document CNV content; all retained network genes are eligible for degree-matched references.
The SCZ GWAS prioritized genes and comparator Catalog genes have different ascertainment; see `input/processed/source_manifest.json`.


In [ ]:
from cnv_overlap.results import resolve_path
for family, run_dir in RUNS.items():
    print("Family:", family)
    display(pd.read_csv(resolve_path(run_dir / "seed_qc.tsv"), sep="\t"))
    display(pd.read_csv(resolve_path(run_dir / "seed_overlap.tsv"), sep="\t").set_index("entity"))
    print("Planned comparisons:", len(pd.read_csv(resolve_path(run_dir / "hypotheses.tsv"), sep="\t")))


## CNV–SCHEMA and CNV–GWAS profiles

One BH family covers all retained CNV–SCHEMA and CNV–GWAS pairs, separately for each null and statistic. Other disorder targets are excluded. Stars indicate BH q < 0.05 for overlap size; empirical q-values are sensitivity diagnostics.


In [ ]:
from cnv_overlap.results import resolve_path
from cnv_overlap.statistics import bh
frames = []
for family, run_dir in RUNS.items():
    frame = pd.read_csv(resolve_path(run_dir / "pair_tests.tsv"), sep="\t").assign(question_family=family)
    frame["target_group"] = np.where(frame.b.isin(SCZ_TARGETS), "scz", "secondary")
    frame["q_empirical"] = np.nan
    for _, idx in frame.groupby(["null", "family", "statistic"]).groups.items():
        frame.loc[idx, "q_empirical"] = bh(frame.loc[idx, "p_empirical"])
    frames.append(frame)
    # COLOC score heatmap, and the same colours labelled with colocalised network gene counts.
    for metric, name in (("ratio", "ratio"), ("ratio_genes", "genes")):
        fig = pair_heatmap(run_dir, family="locus_target", fdr=config.fdr, metric=metric)
        fig.savefig(resolve_path(FIGURES / f"{family}_{name}.png"), dpi=200, bbox_inches="tight")
        fig.savefig(resolve_path(FIGURES / f"{family}_{name}.pdf"), bbox_inches="tight")
        plt.show()
    # Brain analysis includes SCHEMA/GWAS only; no non-SCZ heatmap applies.
tests = pd.concat(frames, ignore_index=True)
primary = tests[(tests["null"] == "netcoloc") & (tests.statistic == "size")].copy()
primary.to_csv(resolve_path(FIGURES / "all_primary_pairs.tsv"), sep="\t", index=False)
with pd.option_context("display.max_rows", None):
    display(primary.sort_values(["question_family", "a", "b"])[["question_family", "a", "b",
        "observed", "null_mean", "excess", "p", "q", "p_empirical", "q_empirical", "status"]])


## Pair enrichment

Test all CNV × SCHEMA and CNV × GWAS candidate sets. Stars mark pairs with
significant size tests (q < config.fdr).

Use GO BP and SynGO through Enrichr's custom-background API, and the frozen
Reactome GMT locally. The background is the propagated network; shared seeds
are excluded for `overlap=remove`. Terms require ≥5 background genes. BH
correction spans all tested pairs and libraries, including zero-hit terms.
These are descriptive annotations after network selection.

orsum groups significant terms and retains their original q-values. Plots,
full tables, term mappings, query/background genes, and API responses are
saved with the analysis. The first run sends gene symbols to Enrichr.


In [ ]:
from cnv_overlap.results import resolve_path
from brain_pair_enrichment import enrich_pairs
from IPython.display import Image
BRAIN_PAIR_ENRICHMENT = enrich_pairs(
    RUNS["combined"], annotation_dir=root / "input/annotations",
    output_root=results_dir(RESULTS_ROOT, "enrichr_cnv_disorder"),
    fdr=config.fdr, backend="enrichr",
)
print((BRAIN_PAIR_ENRICHMENT / "summary.md").read_text())
display(pd.read_csv(resolve_path(BRAIN_PAIR_ENRICHMENT / "tested_pairs.tsv"), sep="\t")[["a", "b", "pair_group", "q", "significant"]])
for figure in ["enrichment_schema", "enrichment_gwas"]:
    display(Image(filename=str(resolve_path(BRAIN_PAIR_ENRICHMENT / f"{figure}_collapsed.png"))))
display(pd.read_csv(resolve_path(BRAIN_PAIR_ENRICHMENT / "enrichment_significant.tsv"), sep="\t").head(40))
print("Complete brain enrichment results:", BRAIN_PAIR_ENRICHMENT)


## Candidate genes and recurrence

Retain thresholded candidates from all pairs and summarize their CNV/target
membership. Recurrence here is descriptive.


In [ ]:
from cnv_overlap.results import resolve_path
display(tests[tests.statistic != "size"].sort_values(["question_family", "statistic", "q"]).head(30))
candidates = pd.concat([pd.read_csv(resolve_path(p / "candidate_genes.tsv"), sep="\t").assign(question_family=f)
                        for f, p in RUNS.items()], ignore_index=True)
candidates.to_csv(resolve_path(FIGURES / "all_candidate_genes.tsv"), sep="\t", index=False)
display(candidates.head(40))
recurrence = candidates.groupby(["entrez", "symbol"]).agg(n_pairs=("a", "size"),
    n_cnvs=("a", "nunique"), n_targets=("b", "nunique")).reset_index()
recurrence.to_csv(resolve_path(FIGURES / "all_pair_descriptive_recurrence.tsv"), sep="\t", index=False)
display(recurrence.sort_values(["n_cnvs", "n_pairs"], ascending=False).head(30))


## Descriptive communities

Partition each connected component separately and summarize candidate-gene
recurrence, degree, and community membership.


In [ ]:
from cnv_overlap.results import resolve_path
from cnv_overlap.interpretation import candidate_network
G, membership = candidate_network(A, nodes, candidates)
membership["symbol"] = membership.entrez.map(symbols)
membership.to_csv(resolve_path(FIGURES / "candidate_communities.tsv"), sep="\t", index=False)
print(f"{G.number_of_nodes()} candidate nodes, {G.number_of_edges()} edges")
display(membership.sort_values(["community", "degree_in_candidate_network"], ascending=[True, False]).head(40))


## Offline annotation

Annotate with explicit GMT snapshots using `cnv-overlap annotate`. Descriptive
ORA uses the tested network universe as its background.


In [ ]:
from cnv_overlap.results import resolve_path
from notebook_analysis import export_versioned_report
GMT_PATHS = [root / "input/annotations" / name for name in
             ["GO_Biological_Process_2026.gmt", "ReactomePathways.gmt"]]
if not all(path.exists() for path in GMT_PATHS):
    print("Annotation snapshots missing; exporting figures and networks only.")
    GMT_PATHS = []
REPORTS = {}
for family, run_dir in RUNS.items():
    REPORTS[family] = export_versioned_report(run_dir, FAMILY_INPUTS[family],
        output_dir=results_dir(RESULTS_ROOT, "cnv_disorder_report"),
        gmt_paths=GMT_PATHS, fdr=config.fdr)
    print(family, REPORTS[family])
    if GMT_PATHS:
        annotation = pd.read_csv(resolve_path(REPORTS[family] / "enrichment/descriptive_ora.tsv"), sep="\t")
        selected_pairs = primary.loc[(primary.question_family == family) & (primary.q < config.fdr), ["a", "b"]]
        displayed = annotation.merge(selected_pairs, on=["a", "b"])
        display(displayed[displayed.q_all_pair_terms < config.fdr].sort_values("q_all_pair_terms").head(40))


In [ ]:
from cnv_overlap.results import finalize_manifest
from cnv_overlap.provenance import file_hash
figure_sources = globals().get("RUNS", {"main": globals().get("RUN")})
finalize_manifest(FIGURES / "manifest.json", {"status": "complete",
    "identity": {"network": NETWORK, "notebook": "cnv_brain_disorder_overlap.ipynb", "notebook_sha256": file_hash(ROOT / "notebooks" / "cnv_brain_disorder_overlap.ipynb")},
    "sources": {name: {"path": str(source.resolve()), "manifest_sha256": file_hash(source / "manifest.json")}
                for name, source in figure_sources.items() if source is not None}})
